In [0]:
data = [
    ("C001", "2024-01-01"),
    ("C001", "2024-01-02"),
    ("C001", "2024-01-04"),
    ("C001", "2024-01-06"),
    ("C002", "2024-01-03"),
    ("C002", "2024-01-05"),
]

df = spark.createDataFrame(data, ["customer_id", "billing_date"])
display(df)

In [0]:
df.dtypes
df=df.withColumn('billing_date',col("billing_date").cast('date'))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_dates = df.groupBy('customer_id').agg(min("billing_date").alias('min_date'),
                                         max("billing_date").alias('max_date'))\
             .withColumn('dates',explode(sequence(col('min_date'),col('max_date'))))

In [0]:
display(df)
display(df_dates)

In [0]:
df_join = df_dates.join(df, ((col('billing_date')==col('dates')) & (df['customer_id']==df_dates['customer_id'])),'left_anti').drop("min_date","max_date")
display(df_join)

In [0]:
window_spec = Window.partitionBy('customer_id').orderBy('dates')
df_lag = df_join.withColumn('lag_date', lag(col('dates')).over(window_spec))\
    .withColumn('id', when(col('lag_date').isNull(), 1)
                  .when(datediff(col('dates'), col('lag_date')) > 1, 1)
                  .otherwise(0))\
    .withColumn('sum_id',sum('id').over(window_spec))\
    .drop('lag_date',"id")
df_result = df_lag.groupBy('customer_id','sum_id').agg(
    min('dates').alias('missing_from'),
    max('dates').alias('missing_to')
)
display(df_result)